# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedfurqan1/FlyRank-MachineLearning/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** One row represents daily search and analytics performance for one content item (content_hash_id) of one client (client_hash_id) on one date (report_date).

**Tables Used:** fact_content_daily_performance_sample (and fact_content_daily_performance), joined with dim_content and dim_clients.

**Time Window:** Feature observation window of March 2026 (2026-03-01 to 2026-03-31), paired with an outcome evaluation window in April 2026 (2026-04-01 to 2026-04-30). June 2026 (2026-06) is reserved as a sealed test set.

**Prediction Target:** Binary indicator predicting a drop (>15%) in organic search impressions in April 2026 compared to March 2026.

**Excluded Field:** trend_direction and trend_pct (they use future window performance and cause direct target leakage).

---



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Context:** client_hash_id, content_hash_id, report_date.       
**Feature:** gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engagement_rate, ga4_data_available.      
**Label:** A derived binary target predicting a >15% drop in organic impressions.       
**Excluded:** trend_direction and trend_pct.   
**Why:** These are calculated using future data, which causes target leakage.



---





## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
MAIN_FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

grain_check = con.execute(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as cnt
    FROM read_parquet('{MAIN_FACT}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()

print("Grain verification:")
print(grain_check)

window_check = con.execute(f"""
    SELECT
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date,
        COUNT(*) AS total_rows
    FROM read_parquet('{MAIN_FACT}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()

print("\nWindow and counts verification:")
print(window_check)

missing_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS null_ga4_flag,
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS null_gsc,
        SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS false_ga4_flag
    FROM read_parquet('{MAIN_FACT}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()

print("\nMissing values verification:")
print(missing_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain verification:
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, cnt]
Index: []


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Window and counts verification:
  start_date   end_date  total_rows
0 2026-03-01 2026-03-31     9841378


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Missing values verification:
   total_rows  null_ga4_flag  null_gsc  false_ga4_flag
0     9841378      3018741.0       0.0       6408671.0


**Grain verification:** The query returned an empty dataframe, proving there are zero duplicate rows. The grain is exactly one client, one content item, and one date.      
**Window and counts:** The data is perfectly bounded from March 1, 2026, to March 31, 2026. The observation window contains exactly 9,841,378 rows.      
**Missing values:** Google Search Console (GSC) data is 100% complete with zero nulls.However, GA4 tracking is mostly absent—over 9.4 million rows have a NULL or FALSE ga4_data_available flag. GA4 features are mostly empty and will require strict handling.

---



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Unbalanced history:** The dataset cannot provide an equal amount of historical context for every entity. As proven below, while the average client has 29.9 days of data in March, some have data for all 31 days, while others have as few as 9 days. This means we cannot assume a perfectly uniform baseline for everyone.

**GSC-only early rows:** We are severely limited to search engine visibility metrics. While Google Search Console (GSC) data is present for all 9,841,378 rows (100%), Google Analytics 4 (GA4) tracking is only available for 413,966 rows (about 4.2%). We are blind to actual on-page user engagement for over 95% of the dataset.

**Window overlaps:** Because the grain is daily, we generate daily prediction rows that look forward to a 30-day monthly outcome. The query below proves that single content items have up to 31 distinct rows in March. A prediction made on March 1 and March 2 for the same content will share 29 days of the exact same future outcome window, meaning the target labels are heavily overlapped and correlated.




In [12]:
MAIN_FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

unbalanced_proof = con.execute(f"""
    SELECT
        MIN(active_days) as min_days_per_client,
        MAX(active_days) as max_days_per_client,
        ROUND(AVG(active_days), 1) as avg_days_per_client
    FROM (
        SELECT client_hash_id, COUNT(DISTINCT report_date) as active_days
        FROM read_parquet('{MAIN_FACT}')
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY client_hash_id
    )
""").df()

gsc_ga4_proof = con.execute(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(gsc_impressions) as rows_with_gsc_data,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as rows_with_ga4_data
    FROM read_parquet('{MAIN_FACT}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()

overlap_proof = con.execute(f"""
    SELECT
        content_hash_id,
        COUNT(report_date) as daily_predictions_made
    FROM read_parquet('{MAIN_FACT}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY content_hash_id
    ORDER BY daily_predictions_made DESC
    LIMIT 3
""").df()

print("--- Proof 1: Unbalanced History ---")
print(unbalanced_proof)

print("\n--- Proof 2: GSC-only Early Rows ---")
print(gsc_ga4_proof)

print("\n--- Proof 3: Window Overlaps ---")
print(overlap_proof)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Proof 1: Unbalanced History ---
   min_days_per_client  max_days_per_client  avg_days_per_client
0                    9                   31                 29.9

--- Proof 2: GSC-only Early Rows ---
   total_rows  rows_with_gsc_data  rows_with_ga4_data
0     9841378             9841378            413966.0

--- Proof 3: Window Overlaps ---
            content_hash_id  daily_predictions_made
0  content_d0dff76c889de68f                      31
1  content_67741cce996cfafa                      31
2  content_2e6360ad20fd7107                      31


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.